In [ ]:
# =============================================
# BASELINE TRAINING - NO REGULARIZATION (FIXED)
# =============================================

print("Installing/importing packages...")
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.models as models
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, 
                            roc_auc_score, cohen_kappa_score, 
                            matthews_corrcoef, balanced_accuracy_score)
import cv2
import json

print("✅ All imports successful")

# =============================================
# BASELINE CONFIGURATION - NO REGULARIZATION
# =============================================
class Config:
    # Dataset - NO LIMITS!
    DATA_PATH = '/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset'
    IMG_SIZE = 224
    BATCH_SIZE = 32
    
    # Training - NO REGULARIZATION
    EPOCHS = 15
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.0      # ❌ NO L2 regularization
    DROPOUT_RATE = 0.0      # ❌ NO Dropout
    NUM_WORKERS = 2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Models to train
    MODELS_TO_TRAIN = ['resnet50', 'densenet121']
    
    # Output
    OUTPUT_DIR = '/kaggle/working/alzheimer_baseline'
    MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')
    LOGS_DIR = os.path.join(OUTPUT_DIR, 'logs')
    PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')

config = Config()

# Create directories
for dir_path in [config.OUTPUT_DIR, config.MODELS_DIR, config.LOGS_DIR, config.PLOTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print(f"📱 Device: {config.DEVICE}")
print(f"📁 Output: {config.OUTPUT_DIR}")
print(f"⚡ Training BASELINE - NO REGULARIZATION")
print(f"   ❌ Dropout: {config.DROPOUT_RATE}")
print(f"   ❌ Weight Decay: {config.WEIGHT_DECAY}")

# =============================================
# DATASET CLASS - USING ALL IMAGES
# =============================================
class AlzheimerMRIDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        
        # Get classes
        self.classes = sorted([d for d in os.listdir(data_dir) 
                              if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i: c for i, c in enumerate(self.classes)}
        
        # Load ALL images
        self.image_paths = []
        self.labels = []
        
        print("📊 Loading FULL dataset...")
        for class_name in tqdm(self.classes, desc="Classes"):
            class_path = os.path.join(data_dir, class_name)
            images = [f for f in os.listdir(class_path) 
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
            for img_name in images:
                self.image_paths.append(os.path.join(class_path, img_name))
                self.labels.append(self.class_to_idx[class_name])
        
        print(f"✅ Loaded {len(self.image_paths)} images from {len(self.classes)} classes")
        
        # Show distribution
        print("\n📈 Class distribution:")
        for i, cls in enumerate(self.classes):
            count = self.labels.count(i)
            print(f"  {cls}: {count} images")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        except:
            image = np.zeros((config.IMG_SIZE, config.IMG_SIZE, 3), dtype=np.uint8)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# =============================================
# BASELINE TRANSFORMS - Minimal augmentation
# =============================================
def get_transforms(img_size=224, train=True):
    if train:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),  # Only basic augmentation
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])

# =============================================
# BASELINE MODELS - NO REGULARIZATION
# =============================================
class BaselineResNet50(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.model = models.resnet50(pretrained=True)
        
        # ❌ NO Dropout - Just linear layer
        in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(in_features, num_classes)
        
        # Unfreeze all layers for baseline
        for param in self.model.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        return self.model(x)

class BaselineDenseNet121(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.model = models.densenet121(pretrained=True)
        
        # ❌ NO Dropout - Just linear layer
        in_features = self.model.classifier.in_features
        self.model.classifier = nn.Linear(in_features, num_classes)
        
        # Unfreeze all layers for baseline
        for param in self.model.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        return self.model(x)

def create_baseline_model(model_name, num_classes=4):
    if model_name == 'resnet50':
        return BaselineResNet50(num_classes)
    elif model_name == 'densenet121':
        return BaselineDenseNet121(num_classes)
    else:
        raise ValueError(f"Unknown model: {model_name}")

# =============================================
# BASELINE TRAINING FUNCTION - NO REGULARIZATION (FIXED)
# =============================================
def train_baseline_model(model_name, train_loader, val_loader, test_loader, class_names):
    print(f"\n{'='*70}")
    print(f"🎯 BASELINE {model_name.upper()} TRAINING (NO REGULARIZATION)")
    print(f"{'='*70}")
    
    # Create model
    model = create_baseline_model(model_name, len(class_names)).to(config.DEVICE)
    
    # Count trainable parameters
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"📊 Parameters: {trainable:,} trainable / {total:,} total")
    
    # Optimizer - NO weight decay
    optimizer = optim.Adam(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY  # ❌ 0.0
    )
    
    # Learning rate scheduler (FIXED: removed verbose parameter)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=3, factor=0.5
    )
    
    criterion = nn.CrossEntropyLoss()
    
    # Training
    best_val_acc = 0
    best_epoch = 0
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    
    for epoch in range(config.EPOCHS):
        # Training
        model.train()
        train_correct = 0
        train_total = 0
        train_loss = 0
        
        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.EPOCHS} [Train]')
        for images, labels in train_bar:
            images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            train_loss += loss.item()
            
            train_bar.set_postfix({
                'Acc': 100.*train_correct/train_total,
                'Loss': loss.item()
            })
        
        train_acc = 100. * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                val_loss += loss.item()
        
        val_acc = 100. * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        
        # Update scheduler
        scheduler.step(val_acc)
        
        # Save history
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        
        print(f"Epoch {epoch+1}: Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")
        print(f"          Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'train_acc': train_acc,
                'val_loss': avg_val_loss,
                'train_loss': avg_train_loss
            }, os.path.join(config.MODELS_DIR, f'best_baseline_{model_name}.pth'))
            print(f"  💾 Saved best model")
    
    # Test evaluation
    checkpoint = torch.load(os.path.join(config.MODELS_DIR, f'best_baseline_{model_name}.pth'))
    model.load_state_dict(checkpoint['model_state_dict'])
    
    model.eval()
    test_correct = 0
    test_total = 0
    test_preds = []
    test_true = []
    test_probs = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()
            
            test_preds.extend(predicted.cpu().numpy())
            test_true.extend(labels.cpu().numpy())
            test_probs.extend(probs.cpu().numpy())
    
    test_acc = 100. * test_correct / test_total
    
    # Calculate comprehensive metrics
    from sklearn.metrics import roc_auc_score
    
    # Per-class AUC
    per_class_auc = {}
    test_probs = np.array(test_probs)
    for i, cls in enumerate(class_names):
        y_true_class = (np.array(test_true) == i).astype(int)
        y_score_class = test_probs[:, i]
        per_class_auc[cls] = roc_auc_score(y_true_class, y_score_class)
    
    # Calculate confusion matrix
    cm = confusion_matrix(test_true, test_preds)
    
    # Calculate per-class accuracy from confusion matrix
    per_class_acc = {}
    for i, cls in enumerate(class_names):
        per_class_acc[cls] = 100.0 * cm[i, i] / np.sum(cm[i]) if np.sum(cm[i]) > 0 else 0
    
    metrics = {
        'model_name': model_name,
        'test_accuracy': test_acc,
        'best_val_accuracy': best_val_acc,
        'best_epoch': best_epoch,
        'cohens_kappa': cohen_kappa_score(test_true, test_preds),
        'matthews_corrcoef': matthews_corrcoef(test_true, test_preds),
        'balanced_accuracy': balanced_accuracy_score(test_true, test_preds),
        'per_class_auc': per_class_auc,
        'per_class_accuracy': per_class_acc,
        'confusion_matrix': cm.tolist(),
        'classification_report': classification_report(test_true, test_preds, 
                                                     target_names=class_names, 
                                                     output_dict=True),
        'training_history': history
    }
    
    print(f"\n✅ BASELINE {model_name.upper()} RESULTS (NO REGULARIZATION):")
    print(f"  Best Val Accuracy: {best_val_acc:.2f}% (epoch {best_epoch})")
    print(f"  Test Accuracy: {test_acc:.2f}%")
    print(f"  Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"  Cohen's Kappa: {metrics['cohens_kappa']:.4f}")
    
    print(f"\n🎯 Per-Class Accuracy:")
    for cls in class_names:
        print(f"  {cls}: {per_class_acc[cls]:.2f}% (AUC: {per_class_auc[cls]:.4f})")
    
    # Save metrics
    metrics_path = os.path.join(config.LOGS_DIR, f'baseline_{model_name}_metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    return metrics

# =============================================
# MAIN PIPELINE - BASELINE
# =============================================
print("\n" + "="*80)
print("🚀 STARTING BASELINE TRAINING - NO REGULARIZATION")
print("="*80)

# Load dataset
print("\n📦 Loading dataset...")
dataset = AlzheimerMRIDataset(config.DATA_PATH)

# Split dataset
indices = list(range(len(dataset)))
train_idx, test_val_idx = train_test_split(
    indices, test_size=0.3, random_state=42, 
    stratify=dataset.labels
)
val_idx, test_idx = train_test_split(
    test_val_idx, test_size=0.5, random_state=42,
    stratify=[dataset.labels[i] for i in test_val_idx]
)

# Create subsets
train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)
test_subset = Subset(dataset, test_idx)

# Apply transforms
train_subset.dataset.transform = get_transforms(config.IMG_SIZE, train=True)
val_subset.dataset.transform = get_transforms(config.IMG_SIZE, train=False)
test_subset.dataset.transform = get_transforms(config.IMG_SIZE, train=False)

# Create dataloaders
train_loader = DataLoader(train_subset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
val_loader = DataLoader(val_subset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
test_loader = DataLoader(test_subset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)

print(f"\n📊 Dataset split:")
print(f"  Train: {len(train_subset)} images ({len(train_subset)/len(dataset)*100:.1f}%)")
print(f"  Val:   {len(val_subset)} images ({len(val_subset)/len(dataset)*100:.1f}%)")
print(f"  Test:  {len(test_subset)} images ({len(test_subset)/len(dataset)*100:.1f}%)")
print(f"🏷️ Classes: {dataset.classes}")

# Train models
print("\n" + "="*80)
print("🤖 TRAINING BASELINE MODELS (NO REGULARIZATION)")
print("="*80)

baseline_results = {}
for model_name in config.MODELS_TO_TRAIN:
    print(f"\n▶️  Starting {model_name}...")
    metrics = train_baseline_model(model_name, train_loader, val_loader, 
                                  test_loader, dataset.classes)
    baseline_results[model_name] = metrics

# ... rest of the code (visualizations, summary, etc.) remains the same

Installing/importing packages...
✅ All imports successful
📱 Device: cuda
📁 Output: /kaggle/working/alzheimer_baseline
⚡ Training BASELINE - NO REGULARIZATION
   ❌ Dropout: 0.0
   ❌ Weight Decay: 0.0

🚀 STARTING BASELINE TRAINING - NO REGULARIZATION

📦 Loading dataset...
📊 Loading FULL dataset...


Classes: 100%|██████████| 4/4 [00:00<00:00,  6.05it/s]


✅ Loaded 33984 images from 4 classes

📈 Class distribution:
  MildDemented: 8960 images
  ModerateDemented: 6464 images
  NonDemented: 9600 images
  VeryMildDemented: 8960 images

📊 Dataset split:
  Train: 23788 images (70.0%)
  Val:   5098 images (15.0%)
  Test:  5098 images (15.0%)
🏷️ Classes: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

🤖 TRAINING BASELINE MODELS (NO REGULARIZATION)

▶️  Starting resnet50...

🎯 BASELINE RESNET50 TRAINING (NO REGULARIZATION)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s]


📊 Parameters: 23,516,228 trainable / 23,516,228 total


Epoch 1/15 [Train]: 100%|██████████| 744/744 [02:23<00:00,  5.17it/s, Acc=79.4, Loss=0.11]


... (57 outputs left out)

Epoch 14/15 [Train]: 100%|██████████| 744/744 [02:21<00:00,  5.26it/s, Acc=100, Loss=0.000696]


Epoch 14: Train Acc: 99.96%, Val Acc: 99.53%
          Train Loss: 0.0019, Val Loss: 0.0129


Epoch 15/15 [Train]: 100%|██████████| 744/744 [02:21<00:00,  5.24it/s, Acc=99.9, Loss=0.000708]


Epoch 15: Train Acc: 99.89%, Val Acc: 99.29%
          Train Loss: 0.0035, Val Loss: 0.0213

✅ BASELINE DENSENET121 RESULTS (NO REGULARIZATION):
  Best Val Accuracy: 99.61% (epoch 13)
  Test Accuracy: 99.47%
  Balanced Accuracy: 0.9951
  Cohen's Kappa: 0.9929

🎯 Per-Class Accuracy:
  MildDemented: 99.93% (AUC: 1.0000)
  ModerateDemented: 100.00% (AUC: 1.0000)
  NonDemented: 99.10% (AUC: 0.9999)
  VeryMildDemented: 99.03% (AUC: 0.9999)
